#  setting us environment and importing libraries

In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')

In [3]:
# Time
start = dt.datetime(2019,6,8)
end = dt.datetime(2019,6,17)
print(start,end)

2019-06-08 00:00:00 2019-06-17 00:00:00


In [4]:
# run this in order to see the list of collection in superstars table
# ab = cursor.superstars
# ab.list_collection_names()

# getting user ids and team ids created in the specified date range

In [6]:
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$gte': start,'$lt': end}},{"sign_up_details":1, "created_at":1}): # end condition
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
users = pd.DataFrame(dic_flattened)
users = users[["_id","created_at","sign_up_details_device_id"]]
users.columns = ["user_id","create_time","device_id"]
len(users)
#users = users[users['user_id']!='5cfddaf7bba72a0018ea594f']

1413

In [7]:
users.sort_values(['device_id','create_time'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
print(len(users))
users.head()

1330


,user_id,create_time,device_id
984,5d04a9bbd06f31001919b313,2019-06-15 08:18:03.095,ffec7de903346c8f23ed41db0fef88b4
610,5d0384bce9771100184fae7e,2019-06-14 11:27:56.873,ffec289af74ccae1ca25393407ac31cb
670,5d03b56fad6fe100184ee962,2019-06-14 14:55:43.452,ffc3825df3835a737ecb9231fecdf5fa
1226,5d052432a473200012719951,2019-06-15 17:00:34.296,ff7d57e925775c3cd211becae504c1ac
1302,5d05acf0a473200012839fbc,2019-06-16 02:44:00.188,ff79e70b5e97a1eb64204420a5d1171a


In [8]:
team_cursor = cursor.superstars.teams
aw_team = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1,'created_at':1,'name':1}): 
    aw_team.append(documents)
dic_flattened = [flatten(d) for d in aw_team]
teams = pd.DataFrame(dic_flattened)
teams = teams[teams["user"].isin(users["user_id"])]
teams = teams[["_id","user",'created_at','name']]
teams.columns = ["team_id", "user_id","team_create_time","team_name"]

In [9]:
teams = teams[teams['team_name']!='sexy daddies']
print(len(teams))
teams.head()

1329


,team_id,user_id,team_create_time,team_name
0,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces
1,5cfafd9e59e7ea40f010d33b,5cfafd9e59e7ea40f010d32f,2019-06-08 00:13:18.230,Prodigious Spartans
2,5cfafe9302e17c437b1d0b16,5cfafe9302e17c437b1d0b0a,2019-06-08 00:17:23.115,Amazing Brawlers
3,5cfb016ee18f9e45c45c3bf6,5cfb016ee18f9e45c45c3bea,2019-06-08 00:29:34.783,Falcon Rockets
4,5cfb02d47e59824ca8c50cb5,5cfb02d47e59824ca8c50ca9,2019-06-08 00:35:32.610,Raging Rockets


# taking out players in each team

In [14]:
player_cursor = cursor.superstars.players
aw_players = []
for documents in player_cursor.find({'created_at': { '$gte': start}},{'team':1,'level':1}):
    aw_players.append(documents)
dic_flattened = [flatten(d) for d in aw_players]
players = pd.DataFrame(dic_flattened)
players = players[players["team"].isin(teams["team_id"])]
players = players[["_id","team",'level']]
players.columns = ["player_id", "team_id",'player_level']

In [15]:
print(len(players))
players.head()

3478


,player_id,team_id,player_level
0,5cfafb541c9cdd40cac99b32,5cfafb541c9cdd40cac99b2e,6
1,5cfafb541c9cdd40cac99b34,5cfafb541c9cdd40cac99b2e,2
2,5cfafcd259e7ea40f010cf72,5cfafb541c9cdd40cac99b2e,1
3,5cfafd9e59e7ea40f010d33f,5cfafd9e59e7ea40f010d33b,5
4,5cfafd9e59e7ea40f010d341,5cfafd9e59e7ea40f010d33b,3


In [16]:
users_team_player = pd.merge(teams,players,on='team_id')
print(len(users_team_player))
users_team_player.head()

3478


,team_id,user_id,team_create_time,team_name,player_id,player_level
0,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b32,6
1,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b34,2
2,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafcd259e7ea40f010cf72,1
3,5cfafd9e59e7ea40f010d33b,5cfafd9e59e7ea40f010d32f,2019-06-08 00:13:18.230,Prodigious Spartans,5cfafd9e59e7ea40f010d33f,5
4,5cfafd9e59e7ea40f010d33b,5cfafd9e59e7ea40f010d32f,2019-06-08 00:13:18.230,Prodigious Spartans,5cfafd9e59e7ea40f010d341,3


# Number of times each user trains

In [17]:
skill_cursor = cursor.superstars.player_skill_logs
aw_skill = []
for documents in skill_cursor.aggregate([{'$unwind':"$details"}, 
                                    {"$match" : {'created_at': {'$gte': start}}},
                                     {"$match" : {"details.type" : 'TRAINING_PROGRESS'}}]):
    aw_skill.append(documents)
dic_flattened = [flatten(d) for d in aw_skill]
players_trained = pd.DataFrame(dic_flattened)
players_trained = players_trained[players_trained["player"].isin(players["player_id"])]
players_trained = players_trained[["created_at","details__id","player"]]
players_trained.columns = ["trained_at","training_id","player_id"]

In [18]:
print(len(players_trained))
players_trained.head()

6120


,trained_at,training_id,player_id
0,2019-06-08 00:23:21.275,5cfafff944b62545a9fe851b,5cfafb541c9cdd40cac99b32
1,2019-06-08 00:23:21.275,5cfb001644b62545a9fe8758,5cfafb541c9cdd40cac99b32
2,2019-06-08 00:23:21.275,5cfb0043a16d3245a3914af0,5cfafb541c9cdd40cac99b32
3,2019-06-08 00:23:21.275,5cfb006281c5d045cb07fe26,5cfafb541c9cdd40cac99b32
4,2019-06-08 00:23:21.275,5cfb0089a16d3245a3914ec3,5cfafb541c9cdd40cac99b32


In [19]:
# only for those who have trained

users_team_player_trained = pd.merge(users_team_player,players_trained, on='player_id')

#users_team_player_trained = users_team_player_trained[(users_team_player_trained['trained_at']-users_team_player_trained['create_time'])<'24:00:00']

In [20]:
print(len(users_team_player_trained))
users_team_player_trained.head()

6120


,team_id,user_id,team_create_time,team_name,player_id,player_level,trained_at,training_id
0,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b32,6,2019-06-08 00:23:21.275,5cfafff944b62545a9fe851b
1,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b32,6,2019-06-08 00:23:21.275,5cfb001644b62545a9fe8758
2,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b32,6,2019-06-08 00:23:21.275,5cfb0043a16d3245a3914af0
3,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b32,6,2019-06-08 00:23:21.275,5cfb006281c5d045cb07fe26
4,5cfafb541c9cdd40cac99b2e,5cfafb541c9cdd40cac99b22,2019-06-08 00:03:32.670,Incredible Aces,5cfafb541c9cdd40cac99b32,6,2019-06-08 00:23:21.275,5cfb0089a16d3245a3914ec3


In [21]:
users_team_player_trained.drop_duplicates(['team_id','player_id'],inplace=True)
len(users_team_player_trained)

1404

In [22]:
total_trained = users_team_player_trained.groupby('user_id').agg({'player_id':'count','team_name':'first'})
total_trained.columns = ['players','team_name']

In [23]:
# uncomment this if you want to include the users who have trained zero players

# total_trained = pd.merge(times_trained_by_user,users[['user_id']],on='user_id',how='right')
# total_trained = total_trained.fillna(0)

the below cell shows that I have trained for 23 times, which is indeed correct.

In [24]:
total_trained.sort_values('players',ascending=False,inplace=True)
print(len(total_trained))
total_trained.head()

433


,players,team_name
user_id,,
5cfbbe28bba72a001861ccbb,10,Super Kings
5d00524eaee56600118f48af,10,Falcon Raiders
5d02a7bb7ed1b000185376c7,9,Legendary Tales
5d03c0ccde1a960011a0b279,9,Fiery Cobras
5d06a9b7a6264f03597bcbac,9,The Great Gladiators


In [25]:
total_trained.describe()

,players
count,433.000000
mean,3.242494
std,1.725046
min,1.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,10.000000


# type of training items used by user_id

## speedup_card_used


In [64]:
c_cards = cursor.superstars.user_collectables_logs # this query will give me how many speedup_cards have been used. 
                                                    #Each tap will be counted as one use of the speedup card
aw_cards = []
for documents in c_cards.aggregate([{'$unwind':"$data"}, 
                                    {"$match" : {'data.quantity': {'$lt': 0},
                                                 'type':'TRAINING_SPEEDUP_CARD',
                                                 'data.reason_type':'SPEEDUP_USE',
                                                 'data.created_at': {'$gte': start}}}]):
    aw_cards.append(documents)
 
dic_flattened = [flatten(d) for d in aw_cards]
cards = pd.DataFrame(dic_flattened)
cards = cards[cards["user"].isin(total_trained.index)]   # total_trained is the number of times each user has trained
cards = cards[["_id","data_created_at",'user']]
cards.columns = ["speedup_card_id", "card_used_at","user_id"]

In [65]:
print(len(cards))
cards.head()

1631


,speedup_card_id,card_used_at,user_id
731,5cfbbeca09e8ab0011cf3dfb,2019-06-13 10:28:19.857,5cfbbe28bba72a001861ccbb
732,5cfbbeca09e8ab0011cf3dfb,2019-06-13 10:28:20.839,5cfbbe28bba72a001861ccbb
733,5cfbbeca09e8ab0011cf3dfb,2019-06-13 10:28:21.774,5cfbbe28bba72a001861ccbb
734,5cfbbeca09e8ab0011cf3dfb,2019-06-13 10:28:23.016,5cfbbe28bba72a001861ccbb
735,5cfbbeca09e8ab0011cf3dfb,2019-06-13 10:28:23.561,5cfbbe28bba72a001861ccbb


In [66]:
cards_used = cards.groupby('user_id')['speedup_card_id'].count().reset_index()
cards_used.columns = ['user_id','number_of_times_speedup_card_used']
print(len(cards_used))
cards_used.sort_values('number_of_times_speedup_card_used',ascending=False,inplace=True)
cards_used.head()

118


,user_id,number_of_times_speedup_card_used
22,5d0104f71052b70011f9efb2,62
19,5d00524eaee56600118f48af,61
28,5d01e67b1052b700111a587e,60
17,5d000f0d2d2aa10018939a06,59
35,5d02c4587ed1b0001854429b,58


## speedup coins used

In [68]:
c_speedup_coins = cursor.superstars.user_collectables_logs
aw_speedup_coins = []
for documents in c_speedup_coins.aggregate([{'$unwind':"$data"},        # this query will give me the the number of times hitcoins has been 
                                {"$match" : {"type":"HARD_CURRENCY",    # to purchase speedup cards
                                  "data.reason_type" : 'SPEEDUP_USE',
                                  'data.quantity': {'$lt': 0},
                                  'data.created_at': {'$gte': start}}}]):
    aw_speedup_coins.append(documents)
    
dic_flattened = [flatten(d) for d in aw_speedup_coins]
speedup_coins = pd.DataFrame(dic_flattened)
speedup_coins = speedup_coins[speedup_coins["user"].isin(total_trained.index)]  # total trained is the number of times each user has trained
speedup_coins = speedup_coins[["_id","data_created_at",'user','data_quantity']]
speedup_coins.columns = ["speedup_coin_id", "speedup_coin_used_at","user_id","coins_used_for_speedup"]

In [69]:
print(len(speedup_coins))
speedup_coins.head()

97


,speedup_coin_id,speedup_coin_used_at,user_id,coins_used_for_speedup
366,5cfb76d8bba72a001841f524,2019-06-09 10:48:12.333,5cfb751909e8ab0011a95b2a,-22
367,5cfb76d8bba72a001841f524,2019-06-09 11:18:08.275,5cfb751909e8ab0011a95b2a,-7
368,5cfb76d8bba72a001841f524,2019-06-09 11:32:51.155,5cfb751909e8ab0011a95b2a,-22
369,5cfb76d8bba72a001841f524,2019-06-09 15:22:56.392,5cfb751909e8ab0011a95b2a,-7
370,5cfb76d8bba72a001841f524,2019-06-09 16:04:59.775,5cfb751909e8ab0011a95b2a,-3


In [70]:
speedupcoins_used = speedup_coins.groupby('user_id')['speedup_coin_id'].count().reset_index()
print(len(speedupcoins_used))
speedupcoins_used.columns = ['user_id','number_of_times_speedup_coins_used']
speedupcoins_used.head()

20


,user_id,number_of_times_speedup_coins_used
0,5cfb751909e8ab0011a95b2a,5
1,5cfd069209e8ab0011320433,8
2,5cfd3895bba72a0018d090ae,10
3,5cff563e50ebd4006236b923,1
4,5cffca72aee56600117bfed3,1


## coins used for instant training

In [72]:
c_instant_coins = cursor.superstars.user_collectables_logs
aw_instant_coins = []
for documents in c_instant_coins.aggregate([{'$unwind':"$data"}, 
                    {"$match" : {"data.reason_type" : 'QUICK_TRAINING',  # the quick training is actually instant training in the game
                                  "type":"HARD_CURRENCY",                # the 'INSTANT' word appers next to "TRAIN"
                                  'data.quantity': {'$lt': 0},           # here this is saved as quick training
                                  'data.created_at': {'$gte': start}}}]):
    aw_instant_coins.append(documents)
    
dic_flattened = [flatten(d) for d in aw_instant_coins]
instant_coins = pd.DataFrame(dic_flattened)
instant_coins = instant_coins[instant_coins["user"].isin(total_trained.index)]
instant_coins = instant_coins[["_id","data_created_at",'user','data_quantity']]
instant_coins.columns = ["instant_coin_id", "instant_coin_used_at","user_id",'coins_used_for_insatnt_train']

In [73]:
print(len(instant_coins))
instant_coins.head()

1542


,instant_coin_id,instant_coin_used_at,user_id,coins_used_for_insatnt_train
701,5cfafe95369f2e43754824f4,2019-06-08 00:27:44.349,5cfafd9e59e7ea40f010d32f,-2
703,5cfb00f744b62545a9fe87ee,2019-06-08 00:36:08.014,5cfafe9302e17c437b1d0b0a,-2
712,5cfb1b941785fd4c87b26c06,2019-06-08 02:21:08.895,5cfb1a816d6da94c8152742f,-1
713,5cfb1b941785fd4c87b26c06,2019-06-08 02:21:24.429,5cfb1a816d6da94c8152742f,-1
714,5cfb1b941785fd4c87b26c06,2019-06-08 02:21:35.013,5cfb1a816d6da94c8152742f,-1


In [74]:
instantcoins_used = instant_coins.groupby('user_id')['instant_coin_id'].count().reset_index()
print(len(instantcoins_used))
instantcoins_used.columns = ['user_id','number_of_times_instant_coins_used']
instantcoins_used.head()

235


,user_id,number_of_times_instant_coins_used
0,5cfafd9e59e7ea40f010d32f,1
1,5cfafe9302e17c437b1d0b0a,1
2,5cfb1a816d6da94c8152742f,5
3,5cfb55d309e8ab00119ba364,1
4,5cfb623809e8ab0011a202f5,21


## coins used to end training once the training starts

In [75]:
c_end_training = cursor.superstars.user_collectables_logs
aw_end_training_coins = []
for documents in c_end_training.aggregate([{'$unwind':"$data"},   # used for ending training once the training has started
                    {"$match" : {"data.reason_type" : 'END_TRAINING', # shown as 'FINISH' right next to 'SPEEDUP', once the training starts
                                  "type":"HARD_CURRENCY",               
                                  'data.quantity': {'$lt': 0},           
                                  'data.created_at': {'$gte': start}}}]):
    aw_end_training_coins.append(documents)
    
dic_flattened = [flatten(d) for d in aw_end_training_coins]
end_training_coins = pd.DataFrame(dic_flattened)
end_training_coins = end_training_coins[end_training_coins["user"].isin(total_trained.index)]
end_training_coins = end_training_coins[["_id","data_created_at",'user','data_quantity']]
end_training_coins.columns = ["end_training_coin_id", "end_coin_used_at","user_id",'coins_used_for_ending_training']

In [76]:
print(len(end_training_coins))
end_training_coins.head()

429


,end_training_coin_id,end_coin_used_at,user_id,coins_used_for_ending_training
303,5cfafe95369f2e43754824f4,2019-06-08 00:27:09.234,5cfafd9e59e7ea40f010d32f,-1
306,5cfb8aca09e8ab0011b3c196,2019-06-08 10:15:38.732,5cfb89a209e8ab0011b32169,-1
307,5cfb8c40bba72a00184c3b65,2019-06-08 12:55:31.101,5cfb8a9409e8ab0011b3a367,-1
308,5cfbc175bba72a0018646bef,2019-06-08 18:05:21.027,5cfbc01fbba72a001863ffba,-2
309,5cfbc175bba72a0018646bef,2019-06-09 08:06:44.928,5cfbc01fbba72a001863ffba,-5


In [77]:
endtrainingcoins_used = end_training_coins.groupby('user_id')['end_training_coin_id'].count().reset_index()
print(len(endtrainingcoins_used))
endtrainingcoins_used.columns = ['user_id','number_of_times_coins_used_to_end_training']
endtrainingcoins_used.sort_values('number_of_times_coins_used_to_end_training',inplace=True,ascending=False)
endtrainingcoins_used.head()

150


,user_id,number_of_times_coins_used_to_end_training
138,5d05e6dfd06f310019531a92,19
49,5d01e67b1052b700111a587e,17
47,5d01ac4daaa8930018f076bd,15
52,5d022eb81052b7001129adf5,14
40,5d00524eaee56600118f48af,12


## merging all the items into one

In [78]:
final1 = pd.merge(total_trained,cards_used,on='user_id',how='outer')
final2 = pd.merge(final1,instantcoins_used,on='user_id',how='outer')
final3 = pd.merge(final2,endtrainingcoins_used,on='user_id',how='outer')
final = pd.merge(final3,speedupcoins_used,on='user_id',how='outer')
final = final.fillna(0)

In [79]:
print(len(final))
final.head()

421


,user_id,times_trained,team_name,number_of_times_speedup_card_used,number_of_times_instant_coins_used,number_of_times_coins_used_to_end_training,number_of_times_speedup_coins_used
0,5cfbbe28bba72a001861ccbb,79,Super Kings,31.0,0.0,5.0,0.0
1,5d00524eaee56600118f48af,65,Falcon Raiders,61.0,12.0,12.0,0.0
2,5cfdc2aa09e8ab0011549b27,51,Chaitu Super Kings,4.0,0.0,0.0,0.0
3,5cfd6078bba72a0018dae255,49,The Goofballs,16.0,19.0,3.0,0.0
4,5cfd069209e8ab0011320433,48,Chennai Champs,58.0,17.0,3.0,8.0


In [92]:
final['clicked_to_train'] = final['times_trained'] - final['number_of_times_instant_coins_used']
final['total_speedup_used'] = final['number_of_times_speedup_card_used'] + final['number_of_times_speedup_coins_used']

In [93]:
final.head()

,user_id,times_trained,number_of_times_instant_coins_used,clicked_to_train,number_of_times_coins_used_to_end_training,number_of_times_speedup_card_used,number_of_times_speedup_coins_used,total_speedup_used
0,5cfbbe28bba72a001861ccbb,79,0.0,79.0,5.0,31.0,0.0,31.0
1,5d00524eaee56600118f48af,65,12.0,53.0,12.0,61.0,0.0,61.0
2,5cfdc2aa09e8ab0011549b27,51,0.0,51.0,0.0,4.0,0.0,4.0
3,5cfd6078bba72a0018dae255,49,19.0,30.0,3.0,16.0,0.0,16.0
4,5cfd069209e8ab0011320433,48,17.0,31.0,3.0,58.0,8.0,66.0


In [94]:
final =final[['user_id','times_trained','number_of_times_instant_coins_used','clicked_to_train','number_of_times_coins_used_to_end_training',
              'total_speedup_used']]